In [1]:
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
SPLIT_MODE = "row"  # "row" reproduces the paper-like protocol; "group" tests unseen materials.
ROW_STRATIFY = True  # Set False for literal unstratified train_test_split.

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

data = pd.read_csv("supplementary_data/data.csv")
continuous_cols = ["O", "N", "SSA", "PV", "RMIC", "Dap", "ID/IG", "CD"]
categorical_cols = ["Anion"]
target_col = "Cs"
data["Anion"] = data["Anion"].map({"SO4": 0, "OTf": 1})
if data["Anion"].isna().any():
    raise ValueError("Unknown Anion value found in the dataset.")
data["Anion"] = data["Anion"].astype(np.int64)

x_cont = data[continuous_cols].to_numpy(dtype=np.float32)
x_cat = data[categorical_cols].to_numpy(dtype=np.int64)
y = data[target_col].to_numpy(dtype=np.float32)
indices = np.arange(len(data))

# The material key excludes CD because each material is measured at multiple CD values.
group_cols = ["Ref", "Cathode", "O", "N", "SSA", "PV", "RMIC", "Dap", "ID/IG", "Anion"]
groups = data[group_cols].astype(str).agg("|".join, axis=1).to_numpy()

if SPLIT_MODE == "row":
    stratify = (
        pd.qcut(y, q=10, labels=False, duplicates="drop")
        if ROW_STRATIFY
        else None
    )
    train_full_idx, test_idx = train_test_split(
        indices, test_size=0.20, random_state=SEED, stratify=stratify
    )
    train_stratify = (
        pd.qcut(y[train_full_idx], q=10, labels=False, duplicates="drop")
        if ROW_STRATIFY
        else None
    )
    train_idx, val_idx = train_test_split(
        train_full_idx, test_size=0.15, random_state=SEED, stratify=train_stratify
    )
elif SPLIT_MODE == "group":
    outer_split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    train_full_local, test_local = next(
        outer_split.split(indices, y, groups=groups)
    )
    train_full_idx = indices[train_full_local]
    test_idx = indices[test_local]
    inner_split = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
    train_local, val_local = next(
        inner_split.split(
            train_full_idx, y[train_full_idx], groups=groups[train_full_idx]
        )
    )
    train_idx = train_full_idx[train_local]
    val_idx = train_full_idx[val_local]
else:
    raise ValueError("SPLIT_MODE must be 'row' or 'group'.")

train_groups = set(groups[train_idx])
val_groups = set(groups[val_idx])
test_groups = set(groups[test_idx])
print(
    f"Rows: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)} | "
    f"material groups: train={len(train_groups)}, val={len(val_groups)}, test={len(test_groups)}"
)
print("Group overlap counts train/val/test:", len(train_groups & val_groups), len(train_groups & test_groups), len(val_groups & test_groups))

# Fit every scaler on the training portion only.
x_scaler = StandardScaler().fit(x_cont[train_idx])
y_scaler = StandardScaler().fit(y[train_idx, None])
x_cont_train = x_scaler.transform(x_cont[train_idx]).astype(np.float32)
x_cont_val = x_scaler.transform(x_cont[val_idx]).astype(np.float32)
x_cont_test = x_scaler.transform(x_cont[test_idx]).astype(np.float32)
y_train = y_scaler.transform(y[train_idx, None]).ravel().astype(np.float32)
y_val = y_scaler.transform(y[val_idx, None]).ravel().astype(np.float32)
y_test = y_scaler.transform(y[test_idx, None]).ravel().astype(np.float32)

def make_loader(x_cont_values, x_cat_values, y_values, shuffle):
    dataset = TensorDataset(
        torch.from_numpy(x_cont_values),
        torch.from_numpy(x_cat_values),
        torch.from_numpy(y_values),
    )
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(dataset, batch_size=20, shuffle=shuffle, generator=generator)

train_loader = make_loader(x_cont_train, x_cat[train_idx], y_train, True)
val_loader = make_loader(x_cont_val, x_cat[val_idx], y_val, False)
test_loader = make_loader(x_cont_test, x_cat[test_idx], y_test, False)


Using device: cpu
Rows: train=424, val=75, test=125 | material groups: train=70, val=46, test=59
Group overlap counts train/val/test: 46 58 40


In [2]:
class FTTransformer(nn.Module):
    def __init__(self, categories, num_continuous, dim=32, depth=4, heads=4, dropout=0.1):
        super().__init__()
        if dim % heads != 0:
            raise ValueError(f"dim={dim} must be divisible by heads={heads}.")

        self.num_continuous = num_continuous
        self.cat_embeds = nn.ModuleList(
            [nn.Embedding(num_categories, dim) for num_categories in categories]
        )
        self.num_weight = nn.Parameter(torch.empty(num_continuous, dim))
        self.num_bias = nn.Parameter(torch.zeros(num_continuous, dim))
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        nn.init.normal_(self.num_weight, mean=0.0, std=dim ** -0.5)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=heads,
            dim_feedforward=dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(dim)
        self.regression_head = nn.Linear(dim, 1)

    def forward(self, x_cat, x_cont):
        if x_cat.ndim != 2 or x_cont.ndim != 2:
            raise ValueError("x_cat and x_cont must both be 2-D tensors.")

        cat_tokens = torch.stack(
            [embedding(x_cat[:, i]) for i, embedding in enumerate(self.cat_embeds)],
            dim=1,
        )
        continuous_tokens = (
            x_cont.unsqueeze(-1) * self.num_weight.unsqueeze(0)
            + self.num_bias.unsqueeze(0)
        )
        cls_tokens = self.cls_token.expand(x_cont.size(0), -1, -1)
        tokens = torch.cat([cls_tokens, cat_tokens, continuous_tokens], dim=1)
        encoded = self.encoder(tokens)
        return self.regression_head(self.norm(encoded[:, 0])).squeeze(-1)


# The SI table reports dim=18 and heads=8 for TabTransformer; that pair is
# invalid for standard PyTorch attention (18 is not divisible by 8). These
# compatible defaults are for the FT-Transformer extension in this notebook.
model = FTTransformer(
    categories=[int(data[column].nunique()) for column in categorical_cols],
    num_continuous=len(continuous_cols),
    dim=32,
    depth=4,
    heads=4,
    dropout=0.1,
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=10
)

best_val_loss = float("inf")
best_epoch = 0
best_state = None
epochs_without_improvement = 0

for epoch in range(300):
    model.train()
    train_loss = 0.0
    for batch_x_cont, batch_x_cat, batch_y in train_loader:
        batch_x_cont = batch_x_cont.to(device)
        batch_x_cat = batch_x_cat.to(device)
        batch_y = batch_y.to(device)
        prediction = model(batch_x_cat, batch_x_cont)
        loss = criterion(prediction, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch_y.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    val_predictions = []
    val_labels = []
    with torch.no_grad():
        for batch_x_cont, batch_x_cat, batch_y in val_loader:
            batch_x_cont = batch_x_cont.to(device)
            batch_x_cat = batch_x_cat.to(device)
            batch_y = batch_y.to(device)
            prediction = model(batch_x_cat, batch_x_cont)
            val_loss += criterion(prediction, batch_y).item() * batch_y.size(0)
            val_predictions.append(prediction.cpu().numpy())
            val_labels.append(batch_y.cpu().numpy())
    val_loss /= len(val_loader.dataset)
    val_predictions = np.concatenate(val_predictions)
    val_labels = np.concatenate(val_labels)
    val_r2 = r2_score(val_labels, val_predictions)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch + 1:03d}/300] | Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val R2: {val_r2:.4f}"
        )
    if epochs_without_improvement >= 30:
        print(f"Early stopping at epoch {epoch + 1}")
        break

model.load_state_dict(best_state)

def predict(loader):
    model.eval()
    predictions = []
    labels = []
    with torch.no_grad():
        for batch_x_cont, batch_x_cat, batch_y in loader:
            predictions.append(model(batch_x_cat.to(device), batch_x_cont.to(device)).cpu().numpy())
            labels.append(batch_y.numpy())
    return np.concatenate(predictions), np.concatenate(labels)

test_predictions_scaled, test_labels_scaled = predict(test_loader)
test_predictions = y_scaler.inverse_transform(test_predictions_scaled[:, None]).ravel()
test_labels = y_scaler.inverse_transform(test_labels_scaled[:, None]).ravel()
test_r2 = r2_score(test_labels, test_predictions)
test_rmse = np.sqrt(mean_squared_error(test_labels, test_predictions))
test_mae = mean_absolute_error(test_labels, test_predictions)
test_mape = np.mean(np.abs((test_labels - test_predictions) / test_labels)) * 100

print(f"Best epoch: {best_epoch}")
print("-------- FT-Transformer --------")
print(f"Test R2   : {test_r2:.4f}")
print(f"Test RMSE : {test_rmse:.4f}")
print(f"Test MAE  : {test_mae:.4f}")
print(f"Test MAPE : {test_mape:.2f}%")


f:\projects\paper-review\.venv\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch [010/300] | Train Loss: 0.2936 | Val Loss: 0.3410 | Val R2: 0.6841
Epoch [020/300] | Train Loss: 0.1719 | Val Loss: 0.2989 | Val R2: 0.7231
Epoch [030/300] | Train Loss: 0.1461 | Val Loss: 0.1561 | Val R2: 0.8554
Epoch [040/300] | Train Loss: 0.1188 | Val Loss: 0.1349 | Val R2: 0.8750
Epoch [050/300] | Train Loss: 0.1030 | Val Loss: 0.1776 | Val R2: 0.8355
Epoch [060/300] | Train Loss: 0.0939 | Val Loss: 0.1309 | Val R2: 0.8788
Epoch [070/300] | Train Loss: 0.0680 | Val Loss: 0.1031 | Val R2: 0.9045
Epoch [080/300] | Train Loss: 0.0600 | Val Loss: 0.1029 | Val R2: 0.9047
Epoch [090/300] | Train Loss: 0.0513 | Val Loss: 0.1086 | Val R2: 0.8994
Epoch [100/300] | Train Loss: 0.0461 | Val Loss: 0.1040 | Val R2: 0.9037
Epoch [110/300] | Train Loss: 0.0374 | Val Loss: 0.1023 | Val R2: 0.9053
Epoch [120/300] | Train Loss: 0.0359 | Val Loss: 0.0910 | Val R2: 0.9157
Epoch [130/300] | Train Loss: 0.0369 | Val Loss: 0.0912 | Val R2: 0.9156
Epoch [140/300] | Train Loss: 0.0367 | Val Loss: 0.